# Medallion Architecture — Notebook 1: Reading & Exploring Data

Welcome to the first notebook in the Medallion Architecture tutorial series! Here we read raw sales data from a CSV file in the workspace `Data` folder, convert it into a Spark DataFrame, and explore and understand the data. Saving it to the **Bronze** layer as a Delta table follows in Notebook 2.

## Roadmap

1. **Define the schema** — declare column names and types explicitly (no `inferSchema`)
2. **Read the CSV** — load `Data/Products.csv` from the workspace folder into a Spark DataFrame
3. **Inspect** — print the schema and count the rows
4. **Explore** — sample rows, distinct countries/products, and the order date range
5. **Analyze** — total sales by country and top 10 products by revenue
6. **Data quality** — check every column for NULLs
7. **Save to Bronze** — handled in **Notebook 2**, which writes the DataFrame as the Delta table `demo.demo_sch.products_bronze`


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, DateType,
)

# An explicit schema is safer than inferSchema: column names and types are
# declared up front, and any value that does not fit becomes NULL instead of
# silently mistyping (or failing) the whole read.
products_schema = StructType([
    StructField("OrderDate",        DateType(),    False),
    StructField("InvoiceNum",       StringType(),  False),
    StructField("ProductKey",       IntegerType(), False),
    StructField("Country",          StringType(),  False),
    StructField("EnglishProductName", StringType(), False),
    StructField("SalesAmount",      DoubleType(),  False),
    StructField("UnitPrice",        DoubleType(),  False),
    StructField("OrderQuantity",    IntegerType(), False),
    StructField("TaxAmt",           DoubleType(),  False),
    StructField("TotalProductCost", DoubleType(),  False),
])

csv_path = "/Workspace/Users/surabhid5@gmail.com/Databricks-Medallion-Architecture/Data/Products.csv"

In [0]:
# Read the CSV from the workspace Data folder into a Spark DataFrame.
# The file uses `file:` scheme so serverless compute can read workspace files.
# OrderDate arrives as dd-MM-yyyy, so we tell the reader how to parse it.
products_df = (
    spark.read
    .option("header", True)
    .option("dateFormat", "dd-MM-yyyy")
    .schema(products_schema)
    .csv(f"file:{csv_path}")
)

print("DataFrame created (lazy — nothing is read until an action runs).")

In [0]:
products_df.printSchema()

row_count = products_df.count()
print(f"\nRow count: {row_count:,}")

In [0]:
# Sample the first 10 rows
display(products_df.limit(10))

# Quick sanity checks: distinct countries, distinct products, and the date range
distinct_countries = [
    r.Country for r in products_df.select("Country").distinct().orderBy("Country").collect()
]
print(f"Distinct countries ({len(distinct_countries)}): {distinct_countries}")
print(f"Distinct products: {products_df.select('EnglishProductName').distinct().count():,}")

display(products_df.select(
    F.min("OrderDate").alias("earliest_order"),
    F.max("OrderDate").alias("latest_order"),
))

In [0]:
sales_by_country = (
    products_df
    .groupBy("Country")
    .agg(
        F.countDistinct("InvoiceNum").alias("num_invoices"),
        F.sum("SalesAmount").alias("total_sales"),
        F.round(F.avg("SalesAmount"), 2).alias("avg_sale"),
    )
    .orderBy(F.desc("total_sales"))
)

display(sales_by_country)

In [0]:
top_products = (
    products_df
    .groupBy("EnglishProductName")
    .agg(
        F.sum("SalesAmount").alias("total_revenue"),
        F.sum("OrderQuantity").alias("units_sold"),
    )
    .orderBy(F.desc("total_revenue"))
    .limit(10)
)

display(top_products)

In [0]:
# Count NULLs in every column — with an explicit schema, a NULL here means
# the raw value could not be parsed into the declared type.
null_counts = products_df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in products_df.columns
])

display(null_counts)

rows_with_nulls = row_count - products_df.dropna(how="any").count()
print(f"Rows containing at least one NULL: {rows_with_nulls:,} of {row_count:,}")

## Recap

* Read the raw CSV from the workspace `Data` folder using an **explicit schema** — no type inference
* Explored the data: sample rows, countries and date range, sales by country, top products by revenue
* Checked data quality: zero NULLs across all columns

**Next up:** Notebook 2 — save this DataFrame as a Delta table in the **Bronze** layer.
